# 03 Train Image Model

Train EfficientNet-B0 with transfer learning for binary deepfake image classification.

In [1]:
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, UnidentifiedImageError
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

DATA_ROOT = Path("../../data/raw/images")
MODEL_DIR = Path("../../models/image")
MODEL_PATH = MODEL_DIR / "new_best_model.pth"
SPLITS = ["train", "val", "test"]
CLASSES = ["real", "fake"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
BATCH_SIZE = 32
NUM_WORKERS = 0
EPOCHS = 10
LEARNING_RATE = 0.0004029469429975904
DROPOUT_RATE = 0.4955353561591949

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
def collect_image_paths(data_root: Path) -> pd.DataFrame:
    rows = []
    for split in SPLITS:
        for class_name in CLASSES:
            class_dir = data_root / split / class_name
            print(f"Scanning {class_dir} | exists={class_dir.exists()}")
            if not class_dir.exists():
                continue
            for path in sorted(class_dir.iterdir()):
                if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                    rows.append({"path": str(path), "split": split, "label": class_name, "label_id": 1 if class_name == "fake" else 0})
    return pd.DataFrame(rows)


df = collect_image_paths(DATA_ROOT)
print(f"Loaded records: {len(df)}")
display(df.head())
display(df.groupby(["split", "label"]).size().unstack(fill_value=0))

Scanning ..\..\data\raw\images\train\real | exists=True
Scanning ..\..\data\raw\images\train\fake | exists=True
Scanning ..\..\data\raw\images\val\real | exists=True
Scanning ..\..\data\raw\images\val\fake | exists=True
Scanning ..\..\data\raw\images\test\real | exists=True
Scanning ..\..\data\raw\images\test\fake | exists=True
Loaded records: 190335


,path,split,label,label_id
0,..\..\data\raw\images\train\real\real_0.jpg,train,real,0
1,..\..\data\raw\images\train\real\real_1.jpg,train,real,0
2,..\..\data\raw\images\train\real\real_10.jpg,train,real,0
3,..\..\data\raw\images\train\real\real_100.jpg,train,real,0
4,..\..\data\raw\images\train\real\real_1000.jpg,train,real,0


label,fake,real
split,,
test,5492,5413
train,70001,70001
val,19641,19787


In [3]:
train_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)


class DeepfakeImageDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        print(f"Dataset initialized with {len(self.dataframe)} images")

    def __len__(self):
        return len(self.dataframe)

    def _load_image(self, path: str):
        try:
            with Image.open(path) as image:
                return image.convert("RGB")
        except (OSError, UnidentifiedImageError) as error:
            print(f"Warning: corrupted image replaced with blank tensor: {path} | {error}")
            return Image.new("RGB", (224, 224), color=(0, 0, 0))

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image = self._load_image(row["path"])
        label = torch.tensor(row["label_id"], dtype=torch.float32)
        if self.transform is not None:
            image = self.transform(image)
        return image, label

In [4]:
train_dataset = DeepfakeImageDataset(df[df["split"] == "train"], transform=train_transform)
val_dataset = DeepfakeImageDataset(df[df["split"] == "val"], transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")
images, labels = next(iter(train_loader))
print(f"Debug train batch images: {images.shape}")
print(f"Debug train batch labels: {labels.shape}")
print(f"Debug train labels: {labels[:10].tolist()}")

Dataset initialized with 140002 images
Dataset initialized with 39428 images
Train size: 140002 | Val size: 39428
Debug train batch images: torch.Size([32, 3, 224, 224])
Debug train batch labels: torch.Size([32])
Debug train labels: [0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0]


In [5]:
def build_model() -> nn.Module:
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
    model = models.efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    # Replace classifier with dropout and linear layer
    model.classifier = nn.Sequential(
        nn.Dropout(p=DROPOUT_RATE),
        nn.Linear(in_features, 1)
    )
    return model


model = build_model().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

print(model.classifier)
print(f"Model output units: {model.classifier[1].out_features}")

Sequential(
  (0): Dropout(p=0.4955353561591949, inplace=False)
  (1): Linear(in_features=1280, out_features=1, bias=True)
)
Model output units: 1


In [6]:
def run_epoch(model, loader, criterion, optimizer=None, phase="train"):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.set_grad_enabled(is_train):
        for batch_idx, (images, labels) in enumerate(loader):
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)

            if batch_idx == 0:
                print(f"{phase} batch images shape: {images.shape}")
                print(f"{phase} batch labels shape: {labels.shape}")

            logits = model(images)
            loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                # Gradient clipping for stability
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            probabilities = torch.sigmoid(logits)
            predictions = (probabilities >= 0.5).float()

            if batch_idx == 0:
                print(f"{phase} logits: {logits[:5].detach().cpu().view(-1).tolist()}")
                print(f"{phase} probabilities: {probabilities[:5].detach().cpu().view(-1).tolist()}")
                print(f"{phase} predictions: {predictions[:5].detach().cpu().view(-1).tolist()}")

            batch_size = images.size(0)
            running_loss += loss.item() * batch_size
            correct += (predictions == labels).sum().item()
            total += batch_size

    epoch_loss = running_loss / max(total, 1)
    epoch_accuracy = correct / max(total, 1)
    return epoch_loss, epoch_accuracy

In [7]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
best_val_loss = float("inf")
history = []

print(f"LR: {LEARNING_RATE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Dropout: {DROPOUT_RATE}")

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    train_loss, train_accuracy = run_epoch(model, train_loader, criterion, optimizer, phase="train")
    val_loss, val_accuracy = run_epoch(model, val_loader, criterion, optimizer=None, phase="val")
    
    scheduler.step()

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
        }
    )

    print(
        f"Epoch {epoch}: "
        f"train_loss={train_loss:.4f}, train_acc={train_accuracy:.4f}, "
        f"val_loss={val_loss:.4f}, val_acc={val_accuracy:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"Saved best model to {MODEL_PATH} with val_loss={best_val_loss:.4f}")

history_df = pd.DataFrame(history)
display(history_df)

LR: 0.0004029469429975904
Batch size: 32
Dropout: 0.4955353561591949

Epoch 1/10
train batch images shape: torch.Size([32, 3, 224, 224])
train batch labels shape: torch.Size([32, 1])
train logits: [-0.18099159002304077, -0.4496857523918152, 0.3786068558692932, 0.5699411034584045, -0.10021071881055832]
train probabilities: [0.4548751711845398, 0.3894354999065399, 0.5935370326042175, 0.6387495994567871, 0.4749682545661926]
train predictions: [0.0, 0.0, 1.0, 1.0, 0.0]
val batch images shape: torch.Size([32, 3, 224, 224])
val batch labels shape: torch.Size([32, 1])
val logits: [-9.124555587768555, -11.736320495605469, -1.9171425104141235, -16.328744888305664, -8.786541938781738]
val probabilities: [0.00010894531442318112, 7.997923603397794e-06, 0.12818054854869843, 8.100580828340753e-08, 0.00015275202167686075]
val predictions: [0.0, 0.0, 0.0, 0.0, 0.0]
Epoch 1: train_loss=0.0684, train_acc=0.9746, val_loss=0.1731, val_acc=0.9557
Saved best model to ..\..\models\image\new_best_model.pth wi

,epoch,train_loss,train_accuracy,val_loss,val_accuracy
0,1,0.068397,0.974579,0.173092,0.955666
1,2,0.039594,0.985122,0.058130,0.980699
2,3,0.031149,0.987922,0.065322,0.980014
3,4,0.025313,0.989900,0.052058,0.984453
4,5,0.019675,0.991772,0.054717,0.984529
5,6,0.015074,0.993907,0.049736,0.986304
6,7,0.010754,0.995543,0.067738,0.986330
7,8,0.008081,0.996757,0.064584,0.987801
8,9,0.005155,0.997950,0.066378,0.987902
9,10,0.003968,0.998429,0.071219,0.988105
